# Setup CDCMS.CIL - Notebook Completo

**CDCMS.CIL**: Concept Drift Handling Based on Clustering in the Model Space for Class-Imbalanced Learning

Este notebook configura e testa o CDCMS.CIL para uso em experimentos comparativos.

## Descobertas Importantes

1. **CDCMS.CIL e um fork do MOA** (nao um plugin) - precisa ser compilado do fonte
2. **Bug no codigo fonte**: arquivo `CDCMS_CIL_GMean_OSUS.java` tem classe com nome errado
3. **moa.DoTask nao funciona**: usar codigo Java direto para avaliacao

---

## PARTE 1: Setup do Ambiente

In [ ]:
# =============================================================================
# CELULA 1.1: Verificar/Instalar Java e Maven
# =============================================================================

import subprocess
import os

print("="*70)
print("VERIFICAR AMBIENTE")
print("="*70)

# Verificar Java
try:
    java_result = subprocess.run(["java", "-version"], capture_output=True, text=True)
    if java_result.returncode == 0:
        print("[OK] Java instalado")
        print(java_result.stderr.split('\n')[0])
    else:
        raise FileNotFoundError()
except FileNotFoundError:
    print("Instalando Java...")
    !apt-get update -qq && apt-get install -y -qq default-jdk

# Verificar Maven
try:
    maven_result = subprocess.run(["mvn", "--version"], capture_output=True, text=True)
    if maven_result.returncode == 0:
        print("[OK] Maven instalado")
    else:
        raise FileNotFoundError()
except FileNotFoundError:
    print("Maven nao encontrado. Instalando...")
    !apt-get update -qq && apt-get install -y -qq maven
    print("[OK] Maven instalado")

print("\n[OK] Ambiente pronto!")

VERIFICAR AMBIENTE
[OK] Java instalado
openjdk version "17.0.17" 2025-10-21
Maven nao encontrado. Instalando...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libapache-pom-java.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../00-libapache-pom-java_18-1_all.deb ...
Unpacking libapache-pom-java (18-1) ...
Selecting previously unselected package libatinject-jsr330-api-java.
Preparing to unpack .../01-libatinject-jsr330-api-java_1.0+ds1-5_all.deb ...
Unpacking libatinject-jsr330-api-java (1.0+ds1-5) ...
Selecting previously unselected package libgeronimo-interceptor-3.0-spec-java.
Preparing to unpack .../02-libgeronimo-interceptor-3.0-spec-java_1.0.1-4fakesync_all.deb ...
Unpacking libgeronimo-interceptor-3.0-spec-java (1

In [ ]:
# =============================================================================
# CELULA 1.2: Definir diretorios
# =============================================================================

from pathlib import Path
import shutil

# Diretorios principais
WORK_DIR = Path('/content')
CDCMS_SRC_DIR = WORK_DIR / 'CDCMS_CIL_src'
CDCMS_JARS_DIR = WORK_DIR / 'cdcms_jars'
DEPS_DIR = WORK_DIR / 'cdcms_all_deps'
ROSE_JARS_DIR = WORK_DIR / 'rose_jars'  # Para MOA-dependencies.jar
BUILD_DIR = WORK_DIR / 'cdcms_build'
TEST_DIR = WORK_DIR / 'cdcms_test_output'
RESULTS_DIR = WORK_DIR / 'cdcms_results'

# Criar diretorios
for d in [CDCMS_SRC_DIR, CDCMS_JARS_DIR, DEPS_DIR, ROSE_JARS_DIR, BUILD_DIR, TEST_DIR, RESULTS_DIR]:
    d.mkdir(exist_ok=True)
    print(f"[OK] {d}")

# Definir JARs importantes
CDCMS_JAR = CDCMS_JARS_DIR / 'cdcms_cil_final.jar'
MOA_DEPS_JAR = ROSE_JARS_DIR / 'MOA-dependencies.jar'

print("\nDiretorios criados!")

[OK] /content/CDCMS_CIL_src
[OK] /content/cdcms_jars
[OK] /content/cdcms_all_deps
[OK] /content/rose_jars
[OK] /content/cdcms_build
[OK] /content/cdcms_test_output
[OK] /content/cdcms_results

Diretorios criados!


## PARTE 2: Clonar e Preparar CDCMS.CIL

In [ ]:
# =============================================================================
# CELULA 2.1: Clonar repositorio CDCMS.CIL
# =============================================================================

import subprocess

print("="*70)
print("CLONAR REPOSITORIO CDCMS.CIL")
print("="*70)

CDCMS_REPO = CDCMS_SRC_DIR / 'CDCMS.CIL'

if CDCMS_REPO.exists():
    print(f"[OK] Repositorio ja existe: {CDCMS_REPO}")
else:
    print("Clonando https://github.com/michaelchiucw/CDCMS.CIL ...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/michaelchiucw/CDCMS.CIL.git"],
        cwd=str(CDCMS_SRC_DIR),
        capture_output=True, text=True, timeout=120
    )
    if CDCMS_REPO.exists():
        print("[OK] Repositorio clonado com sucesso!")
    else:
        print(f"[ERRO] {result.stderr}")

CLONAR REPOSITORIO CDCMS.CIL
Clonando https://github.com/michaelchiucw/CDCMS.CIL ...
[OK] Repositorio clonado com sucesso!


In [ ]:
# =============================================================================
# CELULA 2.2: Corrigir bug no codigo fonte
# =============================================================================

print("="*70)
print("CORRIGIR BUG NO CODIGO FONTE")
print("="*70)

# O arquivo CDCMS_CIL_GMean_OSUS.java contem classe CDCMS_GMean_OSUS
# Java exige que nome do arquivo = nome da classe

SRC_DIR = CDCMS_REPO / 'Implementation' / 'moa' / 'src' / 'main' / 'java'
META_DIR = SRC_DIR / 'moa' / 'classifiers' / 'meta'

problematic_file = META_DIR / 'CDCMS_CIL_GMean_OSUS.java'
correct_file = META_DIR / 'CDCMS_GMean_OSUS.java'

if problematic_file.exists():
    print(f"Arquivo com bug encontrado: {problematic_file.name}")

    # Copiar com nome correto
    shutil.copy(problematic_file, correct_file)
    print(f"[OK] Copiado para: {correct_file.name}")

    # Remover original
    problematic_file.unlink()
    print(f"[OK] Arquivo original removido")
elif correct_file.exists():
    print(f"[OK] Bug ja foi corrigido anteriormente")
else:
    print("[AVISO] Arquivos nao encontrados - verificar estrutura")

CORRIGIR BUG NO CODIGO FONTE
Arquivo com bug encontrado: CDCMS_CIL_GMean_OSUS.java
[OK] Copiado para: CDCMS_GMean_OSUS.java
[OK] Arquivo original removido


## PARTE 3: Baixar Dependencias

In [ ]:
# =============================================================================
# CELULA 3.1: Preparar diretorio de dependencias
# =============================================================================

print("="*70)
print("PREPARAR DIRETORIO DE DEPENDENCIAS")
print("="*70)

# Garantir que o diretorio existe
DEPS_DIR.mkdir(exist_ok=True)

print(f"Diretorio: {DEPS_DIR}")
print("\n[OK] Pronto para baixar dependencias na proxima celula")

PREPARAR DIRETORIO DE DEPENDENCIAS
Diretorio: /content/cdcms_all_deps

[OK] Pronto para baixar dependencias na proxima celula


In [ ]:
# =============================================================================
# CELULA 3.2: Baixar TODAS as dependencias via Maven
# =============================================================================
# O Maven resolve automaticamente TODAS as dependencias transitivas.
# Isso e necessario porque ND4J tem muitas sub-dependencias (~200+ JARs).
# =============================================================================

import subprocess
import time

print("="*70)
print("BAIXAR DEPENDENCIAS VIA MAVEN (COMPLETO)")
print("="*70)

MAVEN_DIR = WORK_DIR / 'maven_resolver'

# Criar diretorio Maven
if MAVEN_DIR.exists():
    shutil.rmtree(MAVEN_DIR)
MAVEN_DIR.mkdir()

# Criar pom.xml com TODAS as dependencias do CDCMS.CIL
pom_content = '''<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 http://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>

    <groupId>temp</groupId>
    <artifactId>cdcms-deps</artifactId>
    <version>1.0</version>
    <packaging>jar</packaging>

    <properties>
        <maven.compiler.source>1.8</maven.compiler.source>
        <maven.compiler.target>1.8</maven.compiler.target>
    </properties>

    <dependencies>
        <!-- Weka -->
        <dependency>
            <groupId>nz.ac.waikato.cms.weka</groupId>
            <artifactId>weka-dev</artifactId>
            <version>3.9.2</version>
        </dependency>

        <!-- SizeOfAg -->
        <dependency>
            <groupId>com.github.fracpete</groupId>
            <artifactId>sizeofag</artifactId>
            <version>1.0.4</version>
        </dependency>

        <!-- Apache Commons Math -->
        <dependency>
            <groupId>org.apache.commons</groupId>
            <artifactId>commons-math3</artifactId>
            <version>3.6.1</version>
        </dependency>

        <!-- ND4J - o pacote completo com todas dependencias nativas -->
        <dependency>
            <groupId>org.nd4j</groupId>
            <artifactId>nd4j-native-platform</artifactId>
            <version>1.0.0-beta7</version>
        </dependency>

        <!-- DeepLearning4J core (inclui ND4J) -->
        <dependency>
            <groupId>org.deeplearning4j</groupId>
            <artifactId>deeplearning4j-core</artifactId>
            <version>1.0.0-beta7</version>
        </dependency>

        <!-- MEKA -->
        <dependency>
            <groupId>net.sf.meka</groupId>
            <artifactId>meka</artifactId>
            <version>1.9.2</version>
        </dependency>

        <!-- JFreeChart -->
        <dependency>
            <groupId>org.jfree</groupId>
            <artifactId>jfreechart</artifactId>
            <version>1.0.19</version>
        </dependency>

    </dependencies>

</project>
'''

pom_path = MAVEN_DIR / 'pom.xml'
with open(pom_path, 'w') as f:
    f.write(pom_content)

print(f"[OK] pom.xml criado")

# Executar Maven
print("\nBaixando dependencias (~200+ JARs)...")
print("Isso pode demorar varios minutos...\n")

start = time.time()

result = subprocess.run(
    [
        "mvn",
        "dependency:copy-dependencies",
        f"-DoutputDirectory={DEPS_DIR}",
        "-DincludeScope=runtime",
        "-Dhttps.protocols=TLSv1.2"
    ],
    cwd=str(MAVEN_DIR),
    capture_output=True,
    text=True,
    timeout=1800  # 30 minutos
)

duration = time.time() - start
print(f"Tempo: {duration:.0f}s")

# Verificar resultado
if result.returncode == 0:
    print("[OK] Maven executado com sucesso!")
else:
    print("[AVISO] Maven retornou erro")
    for line in result.stdout.split('\n'):
        if '[ERROR]' in line or 'BUILD' in line:
            print(f"  {line}")

# Contar JARs
jars = list(DEPS_DIR.glob("*.jar"))
total_size = sum(j.stat().st_size for j in jars)

print(f"\nJARs baixados: {len(jars)}")
print(f"Tamanho total: {total_size/(1024*1024):.1f} MB")

# Verificar pacotes criticos
critical = {'nd4j': False, 'weka': False, 'meka': False, 'commons-math': False}
for jar in jars:
    name = jar.name.lower()
    for pkg in critical:
        if pkg in name:
            critical[pkg] = True

print("\nPacotes criticos:")
for pkg, found in critical.items():
    status = "[OK]" if found else "[X]"
    print(f"  {status} {pkg}")

if len(jars) >= 50:
    print("\n[OK] Dependencias baixadas com sucesso!")
else:
    print("\n[AVISO] Poucos JARs - verifique conexao de rede")

BAIXAR DEPENDENCIAS VIA MAVEN (COMPLETO)
[OK] pom.xml criado

Baixando dependencias (~200+ JARs)...
Isso pode demorar varios minutos...

Tempo: 67s
[OK] Maven executado com sucesso!

JARs baixados: 214
Tamanho total: 917.2 MB

Pacotes criticos:
  [OK] nd4j
  [OK] weka
  [OK] meka
  [OK] commons-math

[OK] Dependencias baixadas com sucesso!


In [ ]:
# =============================================================================
# CELULA 3.3: Baixar MOA-dependencies.jar do ROSE
# =============================================================================
# IMPORTANTE: Este JAR contem nz.ac.waikato.cms.locator.ClassTraversal
# que e necessario para o CDCMS em tempo de execucao
# =============================================================================

import urllib.request
import ssl

print("="*70)
print("BAIXAR MOA-dependencies.jar DO ROSE")
print("="*70)

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

MOA_DEPS_URL = "https://github.com/canoalberto/ROSE/raw/master/MOA-dependencies.jar"
MIN_SIZE = 50 * 1024 * 1024  # 50 MB

if MOA_DEPS_JAR.exists() and MOA_DEPS_JAR.stat().st_size > MIN_SIZE:
    print(f"[OK] MOA-dependencies.jar ja existe ({MOA_DEPS_JAR.stat().st_size/(1024*1024):.1f} MB)")
else:
    print("Baixando MOA-dependencies.jar (~62 MB)...")
    print("(Isso pode demorar alguns minutos)")
    try:
        urllib.request.urlretrieve(MOA_DEPS_URL, MOA_DEPS_JAR)
        if MOA_DEPS_JAR.exists() and MOA_DEPS_JAR.stat().st_size > MIN_SIZE:
            print(f"[OK] Baixado: {MOA_DEPS_JAR.stat().st_size/(1024*1024):.1f} MB")
        else:
            print("[ERRO] Download incompleto")
    except Exception as e:
        print(f"[ERRO] {e}")

# Verificar que contem ClassTraversal
if MOA_DEPS_JAR.exists():
    import subprocess
    result = subprocess.run(
        f'jar tf "{MOA_DEPS_JAR}" | grep -i "ClassTraversal"',
        shell=True, capture_output=True, text=True
    )
    if result.stdout:
        print(f"\n[OK] ClassTraversal encontrado:")
        print(f"  {result.stdout.strip()}")
    else:
        print("\n[AVISO] ClassTraversal nao encontrado no JAR")

BAIXAR MOA-dependencies.jar DO ROSE
Baixando MOA-dependencies.jar (~62 MB)...
(Isso pode demorar alguns minutos)
[OK] Baixado: 61.6 MB

[OK] ClassTraversal encontrado:
  nz/ac/waikato/cms/locator/ClassTraversal.class
nz/ac/waikato/cms/locator/AbstractClassTraversal.class


## PARTE 4: Compilar CDCMS.CIL

In [ ]:
# =============================================================================
# CELULA 4.1: Compilar codigo fonte
# =============================================================================

import subprocess
import time

print("="*70)
print("COMPILAR CDCMS.CIL")
print("="*70)

# Limpar build anterior
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
BUILD_DIR.mkdir()

# Montar classpath COMBINADO (MOA-dependencies.jar + outros JARs)
# MOA-dependencies.jar contem MOA, Weka, nz.ac.waikato.cms.locator
all_jars = list(DEPS_DIR.glob("*.jar"))
non_moa_jars = [j for j in all_jars if 'moa-' not in j.name.lower()]

# MOA_DEPS_JAR PRIMEIRO (tem prioridade)
classpath_parts = []
if MOA_DEPS_JAR.exists():
    classpath_parts.append(str(MOA_DEPS_JAR))
    print(f"[OK] MOA-dependencies.jar: {MOA_DEPS_JAR.stat().st_size/(1024*1024):.1f} MB")
else:
    print("[AVISO] MOA-dependencies.jar nao encontrado - execute celula 3.3")

classpath_parts.extend([str(j) for j in non_moa_jars])
classpath = ":".join(classpath_parts)

print(f"JARs no classpath: {len(classpath_parts)}")

# Listar arquivos Java
SRC_DIR = CDCMS_REPO / 'Implementation' / 'moa' / 'src' / 'main' / 'java'
all_java = list(SRC_DIR.rglob("*.java"))
print(f"\nArquivos Java: {len(all_java)}")

# Criar arquivo de fontes
sources_file = BUILD_DIR / 'sources.txt'
with open(sources_file, 'w') as f:
    for jf in all_java:
        f.write(str(jf) + '\n')

# Compilar
print("\nCompilando (pode demorar)...")

compile_cmd = [
    "javac",
    "-d", str(BUILD_DIR),
    "-cp", classpath,
    "-source", "1.8",
    "-target", "1.8",
    "-Xlint:none",
    "-encoding", "UTF-8",
    f"@{sources_file}"
]

start = time.time()
result = subprocess.run(compile_cmd, capture_output=True, text=True, timeout=600)
compile_time = time.time() - start

print(f"\nTempo: {compile_time:.1f}s")
print(f"Return code: {result.returncode}")

# Contar classes
class_files = list(BUILD_DIR.rglob("*.class"))
print(f"Classes geradas: {len(class_files)}")

if result.returncode == 0:
    print("\n[OK] Compilacao bem-sucedida!")

    # Listar classes CDCMS
    cdcms_classes = [c for c in class_files if 'CDCMS' in c.name]
    print(f"\nClasses CDCMS: {len(cdcms_classes)}")
else:
    print("\n[ERRO] Compilacao falhou")
    print("\n--- ERROS (primeiros 30) ---")
    error_lines = [l for l in result.stderr.split('\n') if l.strip()]
    for line in error_lines[:30]:
        print(line[:100])

    # Identificar pacotes faltantes
    print("\n--- Pacotes faltantes ---")
    missing = set()
    for line in result.stderr.split('\n'):
        if 'package' in line and 'does not exist' in line:
            # Extrair nome do pacote
            parts = line.split('package ')
            if len(parts) > 1:
                pkg = parts[1].split(' ')[0]
                missing.add(pkg)

    for pkg in sorted(missing)[:15]:
        print(f"  - {pkg}")

COMPILAR CDCMS.CIL
[OK] MOA-dependencies.jar: 61.6 MB
JARs no classpath: 215

Arquivos Java: 782

Compilando (pode demorar)...

Tempo: 23.6s
Return code: 0
Classes geradas: 1347

[OK] Compilacao bem-sucedida!

Classes CDCMS: 30


In [ ]:
# =============================================================================
# CELULA 4.2: Criar JAR
# =============================================================================

print("="*70)
print("CRIAR JAR")
print("="*70)

CDCMS_JAR = CDCMS_JARS_DIR / 'cdcms_cil_final.jar'

class_files = list(BUILD_DIR.rglob("*.class"))

if class_files:
    print(f"Empacotando {len(class_files)} classes...")

    jar_cmd = ["jar", "cf", str(CDCMS_JAR), "-C", str(BUILD_DIR), "."]
    result = subprocess.run(jar_cmd, capture_output=True, text=True, timeout=120)

    if CDCMS_JAR.exists():
        print(f"\n[OK] JAR criado: {CDCMS_JAR.name}")
        print(f"Tamanho: {CDCMS_JAR.stat().st_size/(1024*1024):.1f} MB")
    else:
        print(f"[ERRO] {result.stderr}")
else:
    print("[ERRO] Nenhuma classe para empacotar")

CRIAR JAR
Empacotando 1347 classes...

[OK] JAR criado: cdcms_cil_final.jar
Tamanho: 2.1 MB


## PARTE 5: Criar Wrapper Java para Avaliacao

In [ ]:
# =============================================================================
# CELULA 5.1: Criar programa Java para avaliacao
# =============================================================================

print("="*70)
print("CRIAR WRAPPER JAVA PARA AVALIACAO")
print("="*70)

# NOTA: moa.DoTask tem bug com CDCMS, entao usamos codigo Java direto

java_evaluator = '''
import moa.classifiers.Classifier;
import moa.classifiers.meta.CDCMS_CIL_GMean;
import moa.classifiers.meta.CDCMS_CIL;
import moa.streams.ArffFileStream;
import com.yahoo.labs.samoa.instances.Instance;
import java.io.*;
import java.util.*;

public class CDCMSEvaluator {

    public static void main(String[] args) {
        if (args.length < 3) {
            System.out.println("Uso: java CDCMSEvaluator <arff_file> <output_file> <classifier>");
            System.out.println("Classificadores: CDCMS_CIL_GMean, CDCMS_CIL");
            return;
        }

        String arffFile = args[0];
        String outputFile = args[1];
        String classifierName = args[2];

        System.out.println("=== CDCMS Evaluator ===");
        System.out.println("Input: " + arffFile);
        System.out.println("Output: " + outputFile);
        System.out.println("Classifier: " + classifierName);

        try {
            // Criar stream
            ArffFileStream stream = new ArffFileStream();
            stream.arffFileOption.setValue(arffFile);
            stream.prepareForUse();

            // Criar classificador
            Classifier classifier;
            if (classifierName.equals("CDCMS_CIL_GMean")) {
                classifier = new CDCMS_CIL_GMean();
            } else if (classifierName.equals("CDCMS_CIL")) {
                classifier = new CDCMS_CIL();
            } else {
                System.out.println("Classificador desconhecido: " + classifierName);
                return;
            }
            classifier.prepareForUse();

            // Metricas
            int numInstances = 0;
            int correctPredictions = 0;
            List<String> results = new ArrayList<>();
            results.add("instance,accuracy,prediction,actual");

            long startTime = System.currentTimeMillis();

            // Test-then-train
            while (stream.hasMoreInstances()) {
                Instance instance = stream.nextInstance().getData();

                // Prever
                double[] prediction = classifier.getVotesForInstance(instance);
                int predictedClass = 0;
                double maxVote = 0;
                for (int i = 0; i < prediction.length; i++) {
                    if (prediction[i] > maxVote) {
                        maxVote = prediction[i];
                        predictedClass = i;
                    }
                }

                int actualClass = (int) instance.classValue();
                if (predictedClass == actualClass) {
                    correctPredictions++;
                }

                // Treinar
                classifier.trainOnInstance(instance);

                numInstances++;
                double accuracy = (double) correctPredictions / numInstances;
                results.add(numInstances + "," + accuracy + "," + predictedClass + "," + actualClass);

                // Progresso
                if (numInstances % 1000 == 0) {
                    System.out.println("  Processadas: " + numInstances + " | Acc: " + String.format("%.4f", accuracy));
                }
            }

            long endTime = System.currentTimeMillis();
            double finalAccuracy = (double) correctPredictions / numInstances;

            // Salvar resultados
            PrintWriter writer = new PrintWriter(new FileWriter(outputFile));
            for (String line : results) {
                writer.println(line);
            }
            writer.close();

            System.out.println("\\n=== RESULTADOS ===");
            System.out.println("Instancias: " + numInstances);
            System.out.println("Accuracy: " + String.format("%.4f", finalAccuracy));
            System.out.println("Tempo: " + (endTime - startTime) + " ms");
            System.out.println("Resultado salvo em: " + outputFile);

        } catch (Exception e) {
            System.out.println("ERRO: " + e.getMessage());
            e.printStackTrace();
        }
    }
}
'''

# Salvar
evaluator_file = TEST_DIR / 'CDCMSEvaluator.java'
with open(evaluator_file, 'w') as f:
    f.write(java_evaluator)

print(f"[OK] CDCMSEvaluator.java criado")

# Classpath para compilacao do wrapper: CDCMS_JAR + MOA_DEPS_JAR
# (conforme CELULA_TESTAR_CDCMS_FINAL.py)
classpath = f"{CDCMS_JAR}:{MOA_DEPS_JAR}"
print(f"Classpath: {classpath}")

compile_result = subprocess.run(
    ["javac", "-cp", classpath, str(evaluator_file)],
    capture_output=True, text=True, timeout=60
)

if compile_result.returncode == 0:
    print("[OK] CDCMSEvaluator compilado")
else:
    print("[ERRO] Compilacao falhou")
    print(compile_result.stderr[:500])

CRIAR WRAPPER JAVA PARA AVALIACAO
[OK] CDCMSEvaluator.java criado
Classpath: /content/cdcms_jars/cdcms_cil_final.jar:/content/rose_jars/MOA-dependencies.jar
[OK] CDCMSEvaluator compilado


In [ ]:
# =============================================================================
# CELULA 5.1 CORRIGIDA: Criar CDCMSEvaluator v2.0
# =============================================================================
# IMPORTANTE: Esta versao recebe argumentos de linha de comando!
# Substitui a celula 5.1 original do notebook.
#
# Uso: java CDCMSEvaluator <arff_file> <output_file> <classifier>
# Saida: CSV com colunas [instance, accuracy, prediction, actual]
# =============================================================================

import subprocess
from pathlib import Path

print("="*70)
print("CELULA 5.1 CORRIGIDA: CDCMSEvaluator v2.0")
print("="*70)

# Verificar que as variaveis necessarias existem
try:
    _ = CDCMS_JAR
    _ = MOA_DEPS_JAR
    _ = TEST_DIR
except NameError:
    print("[ERRO] Variaveis nao definidas!")
    print("Execute a celula 1.2 primeiro para definir:")
    print("  - CDCMS_JAR")
    print("  - MOA_DEPS_JAR")
    print("  - TEST_DIR")
    raise

# =============================================================================
# Codigo Java do CDCMSEvaluator v2.0
# =============================================================================
java_evaluator_v2 = '''
/**
 * CDCMSEvaluator v2.0
 *
 * IMPORTANTE: Esta versao recebe argumentos de linha de comando!
 *
 * Uso: java CDCMSEvaluator <arff_file> <output_file> <classifier>
 *
 * Saida: CSV com colunas [instance, accuracy, prediction, actual]
 */

import moa.classifiers.Classifier;
import moa.classifiers.meta.CDCMS_CIL_GMean;
import moa.classifiers.meta.CDCMS_CIL;
import moa.streams.ArffFileStream;
import com.yahoo.labs.samoa.instances.Instance;
import java.io.*;
import java.util.*;

public class CDCMSEvaluator {

    public static void main(String[] args) {
        // Verificar argumentos
        if (args.length < 3) {
            System.out.println("Uso: java CDCMSEvaluator <arff_file> <output_file> <classifier>");
            System.out.println();
            System.out.println("Classificadores:");
            System.out.println("  - CDCMS_CIL_GMean");
            System.out.println("  - CDCMS_CIL");
            return;
        }

        String arffFile = args[0];
        String outputFile = args[1];
        String classifierName = args[2];

        System.out.println("=== CDCMS Evaluator v2.0 ===");
        System.out.println("Input:      " + arffFile);
        System.out.println("Output:     " + outputFile);
        System.out.println("Classifier: " + classifierName);

        try {
            // 1. Criar stream
            ArffFileStream stream = new ArffFileStream();
            stream.arffFileOption.setValue(arffFile);
            stream.prepareForUse();

            System.out.println("Atributos:  " + stream.getHeader().numAttributes());
            System.out.println("Classes:    " + stream.getHeader().numClasses());

            // 2. Criar classificador
            Classifier classifier;
            if (classifierName.equals("CDCMS_CIL_GMean")) {
                classifier = new CDCMS_CIL_GMean();
            } else if (classifierName.equals("CDCMS_CIL")) {
                classifier = new CDCMS_CIL();
            } else {
                System.out.println("[ERRO] Classificador desconhecido: " + classifierName);
                return;
            }
            classifier.prepareForUse();

            // 3. Processar stream (Test-then-Train)
            int numInstances = 0;
            int correctPredictions = 0;
            long startTime = System.currentTimeMillis();

            List<String> results = new ArrayList<>();
            results.add("instance,accuracy,prediction,actual");

            System.out.println();
            System.out.println("Processando...");

            while (stream.hasMoreInstances()) {
                Instance instance = stream.nextInstance().getData();

                // TESTE
                double[] prediction = classifier.getVotesForInstance(instance);
                int predictedClass = 0;
                double maxVote = Double.NEGATIVE_INFINITY;
                for (int i = 0; i < prediction.length; i++) {
                    if (prediction[i] > maxVote) {
                        maxVote = prediction[i];
                        predictedClass = i;
                    }
                }

                int actualClass = (int) instance.classValue();
                if (predictedClass == actualClass) {
                    correctPredictions++;
                }
                numInstances++;

                double accuracy = (double) correctPredictions / numInstances;
                results.add(numInstances + "," +
                           String.format("%.6f", accuracy) + "," +
                           predictedClass + "," + actualClass);

                // TREINO
                classifier.trainOnInstance(instance);

                if (numInstances % 2000 == 0) {
                    System.out.println("  " + numInstances + " instancias | Acc: " +
                                     String.format("%.4f", accuracy));
                }
            }

            long endTime = System.currentTimeMillis();
            double finalAccuracy = (double) correctPredictions / numInstances;

            // 4. Salvar CSV
            PrintWriter writer = new PrintWriter(new FileWriter(outputFile));
            for (String line : results) {
                writer.println(line);
            }
            writer.close();

            // 5. Resumo
            System.out.println();
            System.out.println("=== RESULTADO ===");
            System.out.println("Instancias: " + numInstances);
            System.out.println("Accuracy:   " + String.format("%.4f", finalAccuracy));
            System.out.println("Tempo:      " + (endTime - startTime) + " ms");
            System.out.println("Salvo em:   " + outputFile);

        } catch (Exception e) {
            System.out.println("[ERRO] " + e.getMessage());
            e.printStackTrace();
        }
    }
}
'''

# =============================================================================
# Salvar e compilar
# =============================================================================
evaluator_file = TEST_DIR / 'CDCMSEvaluator.java'

# Remover versao antiga se existir
old_class = TEST_DIR / 'CDCMSEvaluator.class'
if old_class.exists():
    old_class.unlink()
    print("[INFO] Removida versao antiga do .class")

# Salvar nova versao
with open(evaluator_file, 'w') as f:
    f.write(java_evaluator_v2)

print(f"[OK] CDCMSEvaluator.java v2.0 criado")

# Compilar
print("\nCompilando...")

classpath = f"{CDCMS_JAR}:{MOA_DEPS_JAR}"

compile_result = subprocess.run(
    ["javac", "-cp", classpath, str(evaluator_file)],
    capture_output=True,
    text=True,
    timeout=60
)

if compile_result.returncode == 0:
    class_file = TEST_DIR / 'CDCMSEvaluator.class'
    if class_file.exists():
        print(f"[OK] Compilado com sucesso!")
        print(f"     {class_file} ({class_file.stat().st_size} bytes)")
    else:
        print("[ERRO] .class nao foi criado")
else:
    print("[ERRO] Compilacao falhou:")
    print(compile_result.stderr[:500])

# =============================================================================
# Teste rapido para verificar que funciona
# =============================================================================
print("\n" + "-"*50)
print("TESTE: Verificar que aceita argumentos")
print("-"*50)

test_cmd = [
    "java", "-Xmx1g",
    "-cp", f"{classpath}:{TEST_DIR}",
    "CDCMSEvaluator"
]

test_result = subprocess.run(test_cmd, capture_output=True, text=True, timeout=10)

if "Uso: java CDCMSEvaluator" in test_result.stdout:
    print("[OK] CDCMSEvaluator v2.0 funcionando!")
    print()
    print("Mensagem de uso:")
    for line in test_result.stdout.strip().split('\n')[:5]:
        print(f"  {line}")
else:
    print("[AVISO] Saida inesperada:")
    print(test_result.stdout[:300])
    if test_result.stderr:
        print("Stderr:", test_result.stderr[:200])

print()
print("="*70)
print("[OK] CDCMSEvaluator v2.0 pronto para uso!")
print("="*70)
print()
print("Agora execute a CELULA 7.5 (Teste Rapido) novamente.")


CELULA 5.1 CORRIGIDA: CDCMSEvaluator v2.0
[INFO] Removida versao antiga do .class
[OK] CDCMSEvaluator.java v2.0 criado

Compilando...
[OK] Compilado com sucesso!
     /content/cdcms_test_output/CDCMSEvaluator.class (4524 bytes)

--------------------------------------------------
TESTE: Verificar que aceita argumentos
--------------------------------------------------
[OK] CDCMSEvaluator v2.0 funcionando!

Mensagem de uso:
  Uso: java CDCMSEvaluator <arff_file> <output_file> <classifier>
  
  Classificadores:
    - CDCMS_CIL_GMean
    - CDCMS_CIL

[OK] CDCMSEvaluator v2.0 pronto para uso!

Agora execute a CELULA 7.5 (Teste Rapido) novamente.


## PARTE 6: Teste Rapido

In [ ]:
# =============================================================================
# CELULA 6.1: Teste rapido com dados sinteticos
# =============================================================================
# Conforme CELULA_TESTAR_CDCMS_FINAL.py - usa apenas CDCMS_JAR + MOA_DEPS_JAR
# =============================================================================

import subprocess
import random
import time

print("="*70)
print("TESTE RAPIDO")
print("="*70)

# Criar arquivo ARFF de teste
test_arff = TEST_DIR / 'test_synthetic.arff'

with open(test_arff, 'w') as f:
    f.write("@relation test_synthetic\n")
    f.write("@attribute a1 numeric\n")
    f.write("@attribute a2 numeric\n")
    f.write("@attribute a3 numeric\n")
    f.write("@attribute class {0,1}\n")
    f.write("@data\n")

    random.seed(42)
    for i in range(3000):
        # Simular drift
        if i < 1000:
            a1 = random.gauss(0, 1)
            a2 = random.gauss(0, 1)
        elif i < 2000:
            a1 = random.gauss(2, 1)
            a2 = random.gauss(2, 1)
        else:
            a1 = random.gauss(-1, 1)
            a2 = random.gauss(-1, 1)

        a3 = random.gauss(0, 1)
        cls = 0 if random.random() < 0.9 else 1  # 90-10 imbalanced
        f.write(f"{a1:.4f},{a2:.4f},{a3:.4f},{cls}\n")

print(f"[OK] Arquivo de teste criado: {test_arff}")
print(f"     Instancias: 3000")

# Classpath de execucao: CDCMS_JAR + MOA_DEPS_JAR
# (MOA_DEPS_JAR e um uber JAR que contem tudo para runtime)
classpath = f"{CDCMS_JAR}:{MOA_DEPS_JAR}"

print(f"\nClasspath:")
print(f"  - {CDCMS_JAR.name}")
print(f"  - {MOA_DEPS_JAR.name}")

# Verificar que arquivos existem
if not CDCMS_JAR.exists():
    print(f"\n[ERRO] {CDCMS_JAR} nao encontrado!")
    print("Execute as celulas 4.1 e 4.2 primeiro")
elif not MOA_DEPS_JAR.exists():
    print(f"\n[ERRO] {MOA_DEPS_JAR} nao encontrado!")
    print("Execute a celula 3.3 primeiro")
else:
    # Executar
    output_file = TEST_DIR / 'test_result.csv'

    cmd = [
        "java", "-Xmx4g",
        "-cp", f"{classpath}:{TEST_DIR}",
        "CDCMSEvaluator",
        str(test_arff),
        str(output_file),
        "CDCMS_CIL_GMean"
    ]

    print("\nExecutando teste...")
    start = time.time()

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

    print(f"\nTempo: {time.time()-start:.1f}s")
    print("\nSaida:")
    print(result.stdout)

    if result.stderr:
        # Filtrar warnings
        errors = [l for l in result.stderr.split('\n') if 'WARNING' not in l and l.strip()]
        if errors:
            print("\nErros:")
            for e in errors[:5]:
                print(f"  {e}")

    # Verificar resultado
    if output_file.exists() and output_file.stat().st_size > 0:
        print("\n" + "*"*50)
        print("*** CDCMS.CIL FUNCIONANDO! ***")
        print("*"*50)

TESTE RAPIDO
[OK] Arquivo de teste criado: /content/cdcms_test_output/test_synthetic.arff
     Instancias: 3000

Classpath:
  - cdcms_cil_final.jar
  - MOA-dependencies.jar

Executando teste...

Tempo: 6.8s

Saida:
=== CDCMS Evaluator v2.0 ===
Input:      /content/cdcms_test_output/test_synthetic.arff
Output:     /content/cdcms_test_output/test_result.csv
Classifier: CDCMS_CIL_GMean
Atributos:  4
Classes:    2

Processando...
  2000 instancias | Acc: 0.8995

=== RESULTADO ===
Instancias: 3000
Accuracy:   0.9037
Tempo:      6009 ms
Salvo em:   /content/cdcms_test_output/test_result.csv


Erros:
  Jan 26, 2026 1:31:24 PM com.github.fommil.netlib.ARPACK <clinit>
  Jan 26, 2026 1:31:24 PM com.github.fommil.netlib.ARPACK <clinit>

**************************************************
*** CDCMS.CIL FUNCIONANDO! ***
**************************************************


## PARTE 7: Testar com Datasets do unified_chunks

In [ ]:
# =============================================================================
# CELULA 7.1: Setup e Constantes
# =============================================================================
# Execute esta celula primeiro para configurar todos os paths e constantes

from pathlib import Path
import json
import os

print("="*70)
print("CELULA 7.1: SETUP E CONSTANTES")
print("="*70)

# -----------------------------------------------------------------------------
# 1. Montar Google Drive
# -----------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# 2. Paths Principais
# -----------------------------------------------------------------------------
# Path base no Drive (AJUSTE SE NECESSARIO)
DRIVE_BASE = Path('/content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid')

# Verificar se existe
if DRIVE_BASE.exists():
    print(f"[OK] DRIVE_BASE: {DRIVE_BASE}")
else:
    print(f"[ERRO] Path nao encontrado: {DRIVE_BASE}")
    # Tentar alternativas
    alternatives = [
        Path('/content/drive/MyDrive/DSL-AG-hybrid'),
        Path('/content/drive/Shareddrives/DSL-AG-hybrid'),
    ]
    for alt in alternatives:
        if alt.exists():
            DRIVE_BASE = alt
            print(f"[OK] Usando alternativa: {DRIVE_BASE}")
            break

# Diretorios de dados
UNIFIED_CHUNKS_DIR = DRIVE_BASE / 'unified_chunks'
EXPERIMENTS_DIR = DRIVE_BASE / 'experiments_unified'

# Diretorios de trabalho no Colab (temporarios)
WORK_DIR = Path('/content')
CDCMS_JARS_DIR = WORK_DIR / 'cdcms_jars'
ROSE_JARS_DIR = WORK_DIR / 'rose_jars'
TEST_DIR = WORK_DIR / 'cdcms_test_output'
TEMP_ARFF_DIR = WORK_DIR / 'cdcms_temp_arff'

# Criar diretorios de trabalho
for d in [CDCMS_JARS_DIR, ROSE_JARS_DIR, TEST_DIR, TEMP_ARFF_DIR]:
    d.mkdir(exist_ok=True)

# -----------------------------------------------------------------------------
# 3. JARs do CDCMS (devem existir das celulas anteriores)
# -----------------------------------------------------------------------------
CDCMS_JAR = CDCMS_JARS_DIR / 'cdcms_cil_final.jar'
MOA_DEPS_JAR = ROSE_JARS_DIR / 'MOA-dependencies.jar'

print(f"\n[INFO] Verificando JARs...")
if CDCMS_JAR.exists():
    print(f"[OK] CDCMS_JAR: {CDCMS_JAR.name} ({CDCMS_JAR.stat().st_size/(1024*1024):.1f} MB)")
else:
    print(f"[AVISO] CDCMS_JAR nao encontrado: {CDCMS_JAR}")
    print("        Execute as celulas 4.1 e 4.2 primeiro!")

if MOA_DEPS_JAR.exists():
    print(f"[OK] MOA_DEPS_JAR: {MOA_DEPS_JAR.name} ({MOA_DEPS_JAR.stat().st_size/(1024*1024):.1f} MB)")
else:
    print(f"[AVISO] MOA_DEPS_JAR nao encontrado: {MOA_DEPS_JAR}")
    print("        Execute a celula 3.3 primeiro!")

# -----------------------------------------------------------------------------
# 4. Configuracoes do Experimento
# -----------------------------------------------------------------------------
# Tamanhos de chunk disponiveis
CHUNK_SIZES = {
    'chunk_500': {'size': 500, 'num_chunks': 24},
    'chunk_1000': {'size': 1000, 'num_chunks': 12},
    # 'chunk_2000': {'size': 2000, 'num_chunks': 6},  # Ainda nao executado
}

# Batches por chunk_size
BATCHES = {
    'chunk_500': ['batch_1', 'batch_2', 'batch_3'],
    'chunk_1000': ['batch_1', 'batch_2', 'batch_3', 'batch_4'],
}

# Janela de holdout para comparacao justa com EGIS
HOLDOUT_WINDOW_SIZE = 100  # Primeiras 100 instancias de cada chunk

# Timeout para execucao do CDCMS (segundos)
CDCMS_TIMEOUT = 600  # 10 minutos

# Datasets a excluir (problematicos)
EXCLUDE_DATASETS = ['IntelLabSensors', 'PokerHand']  # NaN ou muito lentos

# -----------------------------------------------------------------------------
# IMPORTANTE: Datasets MULTICLASSE (CDCMS suporta apenas binario)
# -----------------------------------------------------------------------------
# Referencia: Paper "The Value of Diversity for Dealing with Concept Drift
#             in Class-Imbalanced Data Streams" (Chiu & Minku, IEEE DSAA 2025)
# Citacao: "Covtype and INSECTS were originally multi-class problems. They have
#          been adapted into several versions of binary classification problems"
# -----------------------------------------------------------------------------
MULTICLASS_DATASETS = {
    # LED - 10 classes (digitos 0-9)
    'LED_Abrupt_Simple': 10,
    'LED_Gradual_Simple': 10,
    'LED_Stationary': 10,
    # WAVEFORM - 3 classes
    'WAVEFORM_Abrupt_Simple': 3,
    'WAVEFORM_Gradual_Simple': 3,
    'WAVEFORM_Stationary': 3,
    # CovType - 7 classes (forest cover types)
    'CovType': 7,
    # Shuttle - 7 classes
    'Shuttle': 7,
    # RBF_Stationary - 4 classes
    'RBF_Stationary': 4,
}

def is_multiclass_dataset(dataset_name: str) -> bool:
    """Verifica se dataset e multiclasse (nao suportado pelo CDCMS)."""
    return dataset_name in MULTICLASS_DATASETS

print(f"\n[INFO] Configuracoes:")
print(f"  HOLDOUT_WINDOW_SIZE: {HOLDOUT_WINDOW_SIZE}")
print(f"  CDCMS_TIMEOUT: {CDCMS_TIMEOUT}s")
print(f"  EXCLUDE_DATASETS: {EXCLUDE_DATASETS}")
print(f"  MULTICLASS_DATASETS: {len(MULTICLASS_DATASETS)} (CDCMS suporta apenas binario)")

# -----------------------------------------------------------------------------
# 5. Verificar estrutura de dados
# -----------------------------------------------------------------------------
print(f"\n[INFO] Verificando estrutura de dados...")

for chunk_name, config in CHUNK_SIZES.items():
    chunk_dir = UNIFIED_CHUNKS_DIR / chunk_name
    if chunk_dir.exists():
        datasets = [d.name for d in chunk_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
        print(f"[OK] {chunk_name}: {len(datasets)} datasets")
    else:
        print(f"[X] {chunk_name}: nao encontrado")

print(f"\n[INFO] Verificando experimentos existentes (EGIS)...")
for chunk_name in ['chunk_500', 'chunk_1000', 'chunk_500_penalty', 'chunk_1000_penalty']:
    exp_dir = EXPERIMENTS_DIR / chunk_name
    if exp_dir.exists():
        batches = [d.name for d in exp_dir.iterdir() if d.is_dir() and d.name.startswith('batch')]
        print(f"[OK] {chunk_name}: {len(batches)} batches")
    else:
        print(f"[X] {chunk_name}: nao encontrado")

print("\n" + "="*70)
print("[OK] Setup concluido!")
print("="*70)

CELULA 7.1: SETUP E CONSTANTES
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] DRIVE_BASE: /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid

[INFO] Verificando JARs...
[OK] CDCMS_JAR: cdcms_cil_final.jar (2.1 MB)
[OK] MOA_DEPS_JAR: MOA-dependencies.jar (61.6 MB)

[INFO] Configuracoes:
  HOLDOUT_WINDOW_SIZE: 100
  CDCMS_TIMEOUT: 600s
  EXCLUDE_DATASETS: ['IntelLabSensors', 'PokerHand']
  MULTICLASS_DATASETS: 9 (CDCMS suporta apenas binario)

[INFO] Verificando estrutura de dados...
[OK] chunk_500: 53 datasets
[OK] chunk_1000: 53 datasets
[X] chunk_500_penalty: nao encontrado
[X] chunk_1000_penalty: nao encontrado

[INFO] Verificando experimentos existentes (EGIS)...
[OK] chunk_500: 3 batches
[OK] chunk_1000: 4 batches
[OK] chunk_500_penalty: 3 batches
[OK] chunk_1000_penalty: 4 batches

[OK] Setup concluido!


In [ ]:
# =============================================================================
# CELULA 7.2: Funcoes Auxiliares
# =============================================================================
# Funcoes para conversao de dados e calculo de metricas

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("CELULA 7.2: FUNCOES AUXILIARES")
print("="*70)

def calculate_gmean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Calcula G-Mean (media geometrica dos recalls por classe).

    G-Mean = sqrt(Recall_0 * Recall_1 * ... * Recall_n)
    """
    classes = np.unique(y_true)
    recalls = []

    for cls in classes:
        mask = (y_true == cls)
        if mask.sum() == 0:
            continue
        recall = (y_pred[mask] == cls).sum() / mask.sum()
        recalls.append(recall)

    if len(recalls) == 0:
        return 0.0

    # Media geometrica
    gmean = np.prod(recalls) ** (1.0 / len(recalls))
    return float(gmean)


def calculate_all_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Calcula todas as metricas: G-Mean, F1, F1-weighted, Accuracy.
    """
    metrics = {}

    # G-Mean
    metrics['gmean'] = calculate_gmean(y_true, y_pred)

    # F1-Score (macro para binario, weighted para multiclasse)
    try:
        metrics['f1'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    except:
        metrics['f1'] = 0.0
        metrics['f1_weighted'] = 0.0

    # Accuracy
    try:
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
    except:
        metrics['accuracy'] = 0.0

    return metrics


def load_chunks_from_csv(dataset_path: Path, chunk_size_name: str) -> Tuple[pd.DataFrame, int]:
    """
    Carrega todos os chunks CSV de um dataset e concatena.
    CORRECAO: Converte coluna de classe para inteiros.

    Returns:
        Tuple[DataFrame, num_chunks]
    """
    chunks = sorted(dataset_path.glob("chunk_*.csv"),
                   key=lambda x: int(x.stem.split('_')[1]))

    if not chunks:
        return None, 0

    all_data = []
    for chunk_file in chunks:
        df = pd.read_csv(chunk_file)
        all_data.append(df)

    combined = pd.concat(all_data, ignore_index=True)

    # =========================================================================
    # CORRECAO: Converter coluna de classe para inteiros
    # =========================================================================
    class_col = combined.columns[-1]

    # Se for booleano, converter para int
    if combined[class_col].dtype == bool or str(combined[class_col].dtype) == 'bool':
        combined[class_col] = combined[class_col].astype(int)
        print(f"  [CONVERTIDO] Classe: bool -> int")

    # Se for objeto/string com True/False, converter
    elif combined[class_col].dtype == object:
        try:
            combined[class_col] = combined[class_col].map(
                lambda x: 1 if str(x).lower() in ['true', '1', '1.0'] else 0
            )
            print(f"  [CONVERTIDO] Classe: string -> int")
        except:
            pass

    # Se for float, converter para int
    elif 'float' in str(combined[class_col].dtype):
        combined[class_col] = combined[class_col].astype(int)
        print(f"  [CONVERTIDO] Classe: float -> int")

    return combined, len(chunks)


def create_arff_from_dataframe(df: pd.DataFrame, arff_path: Path, relation_name: str) -> bool:
    """
    Converte DataFrame para formato ARFF.
    CORRECAO: Garante que valores de classe sao inteiros.
    """
    try:
        # Fazer copia para nao modificar original
        df = df.copy()

        # =====================================================================
        # CORRECAO: Garantir que classe e inteiro
        # =====================================================================
        class_col = df.columns[-1]

        # Converter booleanos
        if df[class_col].dtype == bool or str(df[class_col].dtype) == 'bool':
            df[class_col] = df[class_col].astype(int)

        # Converter strings True/False
        elif df[class_col].dtype == object:
            df[class_col] = df[class_col].map(
                lambda x: 1 if str(x).lower() in ['true', '1', '1.0'] else 0
            )

        # Converter float para int
        elif 'float' in str(df[class_col].dtype):
            df[class_col] = df[class_col].astype(int)

        # Obter classes unicas (agora como inteiros)
        unique_classes = sorted(df[class_col].unique())

        with open(arff_path, 'w') as f:
            f.write(f"@relation {relation_name}\n\n")

            # Atributos (todas colunas exceto a ultima = classe)
            for col in df.columns[:-1]:
                if df[col].dtype in ['int64', 'float64', 'int32', 'float32']:
                    f.write(f"@attribute {col} numeric\n")
                elif df[col].dtype == bool:
                    f.write(f"@attribute {col} {{0,1}}\n")
                else:
                    unique_vals = sorted(df[col].dropna().unique())
                    vals_str = ",".join(str(v) for v in unique_vals)
                    f.write(f"@attribute {col} {{{vals_str}}}\n")

            # Classe - usar valores inteiros
            class_str = ",".join(str(int(c)) for c in unique_classes)
            f.write(f"@attribute class {{{class_str}}}\n\n")

            # Dados
            f.write("@data\n")
            for _, row in df.iterrows():
                # Converter cada valor, garantindo que classe e int
                values = []
                for i, v in enumerate(row):
                    if i == len(row) - 1:  # Ultima coluna (classe)
                        values.append(str(int(v)))
                    else:
                        values.append(str(v))
                f.write(",".join(values) + "\n")

        return True

    except Exception as e:
        print(f"[ERRO] Criando ARFF: {e}")
        import traceback
        traceback.print_exc()
        return False


def parse_cdcms_output(output_file: Path) -> Optional[pd.DataFrame]:
    """
    Parseia o arquivo de saida do CDCMSEvaluator.

    Formato esperado: instance,accuracy,prediction,actual
    """
    try:
        df = pd.read_csv(output_file)
        required_cols = ['instance', 'prediction', 'actual']

        # Verificar colunas
        for col in required_cols:
            if col not in df.columns:
                print(f"[ERRO] Coluna '{col}' nao encontrada no output")
                return None

        return df
    except Exception as e:
        print(f"[ERRO] Parseando output: {e}")
        return None


def calculate_metrics_per_chunk(
    predictions_df: pd.DataFrame,
    chunk_size: int,
    holdout_window: int = 100
) -> List[Dict]:
    """
    Calcula metricas por chunk a partir das predicoes do CDCMS.

    Para cada chunk, calcula:
    - Metricas prequential (chunk completo)
    - Metricas holdout (primeiras N instancias)

    Args:
        predictions_df: DataFrame com colunas [instance, prediction, actual]
        chunk_size: Tamanho do chunk (500 ou 1000)
        holdout_window: Numero de instancias para holdout (default: 100)

    Returns:
        Lista de dicts com metricas por chunk
    """
    total_instances = len(predictions_df)
    num_chunks = total_instances // chunk_size

    results = []

    for chunk_idx in range(num_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = start_idx + chunk_size

        # Dados do chunk completo
        chunk_data = predictions_df.iloc[start_idx:end_idx]
        y_true = chunk_data['actual'].values.astype(int)
        y_pred = chunk_data['prediction'].values.astype(int)

        # Metricas prequential (chunk completo)
        prequential_metrics = calculate_all_metrics(y_true, y_pred)

        chunk_result = {
            'chunk': chunk_idx,
            'instances_in_chunk': len(chunk_data),
            # Metricas prequential
            'prequential_gmean': prequential_metrics['gmean'],
            'prequential_f1': prequential_metrics['f1'],
            'prequential_f1_weighted': prequential_metrics['f1_weighted'],
            'prequential_accuracy': prequential_metrics['accuracy'],
        }

        # Metricas holdout (apenas para chunk >= 1)
        # No chunk 0, o modelo esta aprendendo do zero
        if chunk_idx >= 1:
            # Usar primeiras N instancias do chunk (holdout window)
            holdout_end = min(start_idx + holdout_window, end_idx)
            holdout_data = predictions_df.iloc[start_idx:holdout_end]

            y_true_holdout = holdout_data['actual'].values.astype(int)
            y_pred_holdout = holdout_data['prediction'].values.astype(int)

            holdout_metrics = calculate_all_metrics(y_true_holdout, y_pred_holdout)

            chunk_result['holdout_gmean'] = holdout_metrics['gmean']
            chunk_result['holdout_f1'] = holdout_metrics['f1']
            chunk_result['holdout_f1_weighted'] = holdout_metrics['f1_weighted']
            chunk_result['holdout_accuracy'] = holdout_metrics['accuracy']
            chunk_result['holdout_window_size'] = holdout_end - start_idx
        else:
            # Chunk 0 - sem holdout (modelo iniciando)
            chunk_result['holdout_gmean'] = None
            chunk_result['holdout_f1'] = None
            chunk_result['holdout_f1_weighted'] = None
            chunk_result['holdout_accuracy'] = None
            chunk_result['holdout_window_size'] = 0
            chunk_result['note'] = 'Chunk 0 - modelo iniciando do zero'

        results.append(chunk_result)

    return results


print("[OK] Funcoes auxiliares carregadas (v2.1 - com fix boolean->int):")
print("  - calculate_gmean(y_true, y_pred)")
print("  - calculate_all_metrics(y_true, y_pred)")
print("  - load_chunks_from_csv(dataset_path, chunk_size_name) [CORRIGIDO]")
print("  - create_arff_from_dataframe(df, arff_path, relation_name) [CORRIGIDO]")
print("  - parse_cdcms_output(output_file)")
print("  - calculate_metrics_per_chunk(predictions_df, chunk_size, holdout_window)")

CELULA 7.2: FUNCOES AUXILIARES
[OK] Funcoes auxiliares carregadas (v2.1 - com fix boolean->int):
  - calculate_gmean(y_true, y_pred)
  - calculate_all_metrics(y_true, y_pred)
  - load_chunks_from_csv(dataset_path, chunk_size_name) [CORRIGIDO]
  - create_arff_from_dataframe(df, arff_path, relation_name) [CORRIGIDO]
  - parse_cdcms_output(output_file)
  - calculate_metrics_per_chunk(predictions_df, chunk_size, holdout_window)


In [ ]:
# =============================================================================
# CELULA 7.3: Verificar datasets disponiveis
# =============================================================================

print("="*70)
print("VERIFICAR DATASETS DISPONIVEIS")
print("="*70)

# Tentar diferentes locais possiveis
possible_paths = [
    Path('/content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/experiments_unified/chunk_500'),
]

CHUNKS_DIR = None
for p in possible_paths:
    if p.exists():
        CHUNKS_DIR = p
        break

if CHUNKS_DIR:
    datasets = sorted([d.name for d in CHUNKS_DIR.iterdir() if d.is_dir()])
    print(f"[OK] Encontrado: {CHUNKS_DIR}")
    print(f"\nDatasets disponiveis ({len(datasets)}):")
    for d in datasets[:15]:
        print(f"  - {d}")
    if len(datasets) > 15:
        print(f"  ... e mais {len(datasets)-15}")
else:
    print("[AVISO] Diretorio de chunks nao encontrado")
    print("Faca upload dos dados ou monte o Google Drive")

VERIFICAR DATASETS DISPONIVEIS
[OK] Encontrado: /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/experiments_unified/chunk_500

Datasets disponiveis (3):
  - batch_1
  - batch_2
  - batch_3


In [ ]:
# =============================================================================
# CELULA 7.3: Funcao Principal run_cdcms_on_dataset
# =============================================================================
# Funcao principal que executa o CDCMS em um dataset

import subprocess
import time
from datetime import datetime

print("="*70)
print("CELULA 7.3: FUNCAO PRINCIPAL run_cdcms_on_dataset")
print("="*70)

def run_cdcms_on_dataset(
    dataset_name: str,
    chunk_size_name: str,  # 'chunk_500' ou 'chunk_1000'
    batch_name: str,       # 'batch_1', 'batch_2', etc.
    classifier: str = "CDCMS_CIL_GMean",
    timeout: int = None,
    save_results: bool = True,
    verbose: bool = True
) -> Optional[Dict]:
    """
    Executa CDCMS em um dataset e calcula metricas por chunk.

    Args:
        dataset_name: Nome do dataset (ex: 'SEA_Abrupt_Simple')
        chunk_size_name: 'chunk_500' ou 'chunk_1000'
        batch_name: 'batch_1', 'batch_2', etc.
        classifier: 'CDCMS_CIL_GMean' ou 'CDCMS_CIL'
        timeout: Timeout em segundos (default: CDCMS_TIMEOUT)
        save_results: Se deve salvar resultados no formato compativel
        verbose: Se deve imprimir mensagens de progresso

    Returns:
        Dict com resultados ou None se falhar
    """
    if timeout is None:
        timeout = CDCMS_TIMEOUT

    # -------------------------------------------------------------------------
    # 1. Verificar paths
    # -------------------------------------------------------------------------
    chunk_config = CHUNK_SIZES.get(chunk_size_name)
    if chunk_config is None:
        print(f"[ERRO] chunk_size_name invalido: {chunk_size_name}")
        return None

    chunk_size = chunk_config['size']

    # Path dos dados (chunks CSV)
    data_path = UNIFIED_CHUNKS_DIR / chunk_size_name / dataset_name
    if not data_path.exists():
        print(f"[ERRO] Dataset nao encontrado: {data_path}")
        return None

    # Path dos resultados EGIS (para salvar ao lado)
    egis_results_path = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset_name

    if verbose:
        print(f"\n--- {dataset_name} ({chunk_size_name}/{batch_name}) ---")

    # -------------------------------------------------------------------------
    # 2. Carregar e converter dados
    # -------------------------------------------------------------------------
    df, num_chunks = load_chunks_from_csv(data_path, chunk_size_name)
    if df is None:
        print(f"[ERRO] Falha ao carregar chunks")
        return None

    if verbose:
        print(f"  Instancias: {len(df)} | Chunks: {num_chunks} | Features: {len(df.columns)-1}")

    # Criar ARFF temporario
    arff_file = TEMP_ARFF_DIR / f"{dataset_name}_{chunk_size_name}.arff"
    if not create_arff_from_dataframe(df, arff_file, dataset_name):
        print(f"[ERRO] Falha ao criar ARFF")
        return None

    # -------------------------------------------------------------------------
    # 3. Executar CDCMS
    # -------------------------------------------------------------------------
    output_file = TEST_DIR / f"{dataset_name}_{classifier}_output.csv"

    # Limpar arquivo anterior se existir
    if output_file.exists():
        output_file.unlink()

    # Classpath
    classpath = f"{CDCMS_JAR}:{MOA_DEPS_JAR}"

    # Comando
    cmd = [
        "java", "-Xmx4g",
        "-cp", f"{classpath}:{TEST_DIR}",
        "CDCMSEvaluator",
        str(arff_file),
        str(output_file),
        classifier
    ]

    if verbose:
        print(f"  Executando {classifier}...")

    start_time = time.time()

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        execution_time = time.time() - start_time

        if result.returncode != 0:
            if verbose:
                print(f"  [ERRO] Return code: {result.returncode}")
                if result.stderr:
                    errors = [l for l in result.stderr.split('\n') if 'Exception' in l or 'Error' in l][:3]
                    for e in errors:
                        print(f"    {e[:80]}")
            return None

    except subprocess.TimeoutExpired:
        if verbose:
            print(f"  [TIMEOUT] Excedeu {timeout}s")
        return None
    except Exception as e:
        if verbose:
            print(f"  [ERRO] {e}")
        return None

    # -------------------------------------------------------------------------
    # 4. Parsear output e calcular metricas
    # -------------------------------------------------------------------------
    if not output_file.exists() or output_file.stat().st_size == 0:
        if verbose:
            print(f"  [ERRO] Arquivo de saida vazio ou inexistente")
        return None

    predictions_df = parse_cdcms_output(output_file)
    if predictions_df is None:
        return None

    # Calcular metricas por chunk
    chunk_metrics = calculate_metrics_per_chunk(
        predictions_df,
        chunk_size,
        HOLDOUT_WINDOW_SIZE
    )

    # -------------------------------------------------------------------------
    # 5. Calcular medias gerais
    # -------------------------------------------------------------------------
    # Media prequential (todos os chunks)
    avg_prequential_gmean = np.mean([m['prequential_gmean'] for m in chunk_metrics])
    avg_prequential_f1 = np.mean([m['prequential_f1'] for m in chunk_metrics])

    # Media holdout (chunks >= 1 apenas)
    holdout_metrics = [m for m in chunk_metrics if m['holdout_gmean'] is not None]
    if holdout_metrics:
        avg_holdout_gmean = np.mean([m['holdout_gmean'] for m in holdout_metrics])
        avg_holdout_f1 = np.mean([m['holdout_f1'] for m in holdout_metrics])
    else:
        avg_holdout_gmean = None
        avg_holdout_f1 = None

    if verbose:
        print(f"  [OK] Tempo: {execution_time:.1f}s")
        print(f"       Prequential G-Mean: {avg_prequential_gmean:.4f}")
        if avg_holdout_gmean is not None:
            print(f"       Holdout G-Mean:     {avg_holdout_gmean:.4f}")

    # -------------------------------------------------------------------------
    # 6. Preparar resultado
    # -------------------------------------------------------------------------
    result_data = {
        'dataset': dataset_name,
        'chunk_size_name': chunk_size_name,
        'chunk_size': chunk_size,
        'batch': batch_name,
        'classifier': classifier,
        'total_instances': len(df),
        'num_chunks': len(chunk_metrics),
        'num_features': len(df.columns) - 1,
        'num_classes': len(df[df.columns[-1]].unique()),
        'execution_time_seconds': execution_time,
        'holdout_window_size': HOLDOUT_WINDOW_SIZE,
        # Medias
        'avg_prequential_gmean': avg_prequential_gmean,
        'avg_prequential_f1': avg_prequential_f1,
        'avg_holdout_gmean': avg_holdout_gmean,
        'avg_holdout_f1': avg_holdout_f1,
        # Metricas por chunk
        'chunk_metrics': chunk_metrics,
        # Timestamp
        'executed_at': datetime.now().isoformat()
    }

    # -------------------------------------------------------------------------
    # 7. Salvar resultados (se solicitado)
    # -------------------------------------------------------------------------
    if save_results and egis_results_path.exists():
        cdcms_results_dir = egis_results_path / 'cdcms_results'
        cdcms_results_dir.mkdir(exist_ok=True)

        # Salvar chunk_metrics.json
        metrics_file = cdcms_results_dir / 'chunk_metrics.json'
        with open(metrics_file, 'w') as f:
            json.dump(chunk_metrics, f, indent=2)

        # Salvar run_config.json
        config_data = {
            'classifier': classifier,
            'chunk_size': chunk_size,
            'chunk_size_name': chunk_size_name,
            'batch': batch_name,
            'holdout_window_size': HOLDOUT_WINDOW_SIZE,
            'total_instances': len(df),
            'num_chunks': len(chunk_metrics),
            'execution_time_seconds': execution_time,
            'executed_at': datetime.now().isoformat(),
            'avg_prequential_gmean': avg_prequential_gmean,
            'avg_holdout_gmean': avg_holdout_gmean
        }
        config_file = cdcms_results_dir / 'run_config.json'
        with open(config_file, 'w') as f:
            json.dump(config_data, f, indent=2)

        # Copiar output bruto
        import shutil
        raw_output_file = cdcms_results_dir / 'cdcms_raw_output.csv'
        shutil.copy(output_file, raw_output_file)

        if verbose:
            print(f"  [SALVO] {cdcms_results_dir}")

    return result_data


print("[OK] Funcao run_cdcms_on_dataset() definida!")
print("\nExemplo de uso:")
print('  result = run_cdcms_on_dataset("SEA_Abrupt_Simple", "chunk_500", "batch_1")')

CELULA 7.3: FUNCAO PRINCIPAL run_cdcms_on_dataset
[OK] Funcao run_cdcms_on_dataset() definida!

Exemplo de uso:
  result = run_cdcms_on_dataset("SEA_Abrupt_Simple", "chunk_500", "batch_1")


In [ ]:
# =============================================================================
# CELULA 7.4: Funcoes de Execucao em Lote
# =============================================================================
# Funcoes para executar CDCMS em multiplos datasets

print("="*70)
print("CELULA 7.4: FUNCOES DE EXECUCAO EM LOTE")
print("="*70)

def get_datasets_for_batch(chunk_size_name: str, batch_name: str) -> List[str]:
    """
    Retorna lista de datasets disponiveis para um batch.
    Baseado nos experimentos EGIS existentes.
    """
    batch_path = EXPERIMENTS_DIR / chunk_size_name / batch_name

    if not batch_path.exists():
        print(f"[AVISO] Batch nao encontrado: {batch_path}")
        return []

    datasets = []
    for d in batch_path.iterdir():
        if d.is_dir() and not d.name.startswith('.') and d.name not in ['desktop.ini']:
            # Verificar se tem resultados EGIS (run_1)
            run_dir = d / 'run_1'
            if run_dir.exists():
                datasets.append(d.name)

    # Filtrar datasets problematicos
    datasets = [d for d in datasets if d not in EXCLUDE_DATASETS]

    return sorted(datasets)


def run_cdcms_batch(
    chunk_size_name: str,
    batch_name: str,
    max_datasets: int = None,
    skip_existing: bool = True
) -> pd.DataFrame:
    """
    Executa CDCMS em todos os datasets de um batch.

    Args:
        chunk_size_name: 'chunk_500' ou 'chunk_1000'
        batch_name: 'batch_1', 'batch_2', etc.
        max_datasets: Limite de datasets (None = todos)
        skip_existing: Pular datasets que ja tem resultados CDCMS

    Returns:
        DataFrame com resultados (incluindo N/A para multiclasse)
    """
    print("="*70)
    print(f"EXECUTAR CDCMS: {chunk_size_name} / {batch_name}")
    print("="*70)

    datasets = get_datasets_for_batch(chunk_size_name, batch_name)

    if max_datasets:
        datasets = datasets[:max_datasets]

    print(f"Datasets a processar: {len(datasets)}")

    results = []
    skipped = 0
    failed = 0
    multiclass_skipped = 0

    for i, dataset in enumerate(datasets, 1):
        # =====================================================================
        # VERIFICAR SE E MULTICLASSE (CDCMS suporta apenas binario)
        # =====================================================================
        if is_multiclass_dataset(dataset):
            num_classes = MULTICLASS_DATASETS[dataset]
            print(f"[{i}/{len(datasets)}] {dataset} - N/A (multiclasse: {num_classes} classes)")

            # Registrar como N/A
            results.append({
                'dataset': dataset,
                'chunk_size': chunk_size_name,
                'batch': batch_name,
                'status': 'N/A',
                'prequential_gmean': None,
                'holdout_gmean': None,
                'prequential_f1': None,
                'holdout_f1': None,
                'time_seconds': 0,
                'num_chunks': 0,
                'note': f'Multiclass ({num_classes} classes) - CDCMS binary only'
            })
            multiclass_skipped += 1
            continue

        # Verificar se ja existe
        if skip_existing:
            cdcms_dir = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset / 'cdcms_results'
            if cdcms_dir.exists() and (cdcms_dir / 'chunk_metrics.json').exists():
                print(f"[{i}/{len(datasets)}] {dataset} - SKIP (ja existe)")
                skipped += 1
                continue

        print(f"[{i}/{len(datasets)}]", end="")
        result = run_cdcms_on_dataset(dataset, chunk_size_name, batch_name)

        if result:
            results.append({
                'dataset': dataset,
                'chunk_size': chunk_size_name,
                'batch': batch_name,
                'status': 'OK',
                'prequential_gmean': result['avg_prequential_gmean'],
                'holdout_gmean': result['avg_holdout_gmean'],
                'prequential_f1': result['avg_prequential_f1'],
                'holdout_f1': result['avg_holdout_f1'],
                'time_seconds': result['execution_time_seconds'],
                'num_chunks': result['num_chunks'],
                'note': ''
            })
        else:
            failed += 1

    print("\n" + "="*70)
    print(f"RESUMO: {chunk_size_name} / {batch_name}")
    print("="*70)
    print(f"  Processados com sucesso: {len([r for r in results if r.get('status') == 'OK'])}")
    print(f"  Pulados (existentes): {skipped}")
    print(f"  Multiclasse (N/A): {multiclass_skipped}")
    print(f"  Falhas: {failed}")

    if results:
        df = pd.DataFrame(results)

        # Filtrar apenas resultados validos (excluir N/A) para calcular medias
        valid_df = df[df['status'] == 'OK']

        if not valid_df.empty:
            print(f"\n  Media Prequential G-Mean: {valid_df['prequential_gmean'].mean():.4f}")
            if valid_df['holdout_gmean'].notna().any():
                print(f"  Media Holdout G-Mean:     {valid_df['holdout_gmean'].mean():.4f}")

        return df

    return pd.DataFrame()


def run_cdcms_all_batches(chunk_size_name: str, skip_existing: bool = True) -> pd.DataFrame:
    """
    Executa CDCMS em todos os batches de um chunk_size.
    """
    all_results = []

    batches = BATCHES.get(chunk_size_name, [])

    for batch in batches:
        df = run_cdcms_batch(chunk_size_name, batch, skip_existing=skip_existing)
        if not df.empty:
            all_results.append(df)

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()


print("[OK] Funcoes de execucao em lote carregadas:")
print("  - get_datasets_for_batch(chunk_size_name, batch_name)")
print("  - run_cdcms_batch(chunk_size_name, batch_name)")
print("  - run_cdcms_all_batches(chunk_size_name)")

CELULA 7.4: FUNCOES DE EXECUCAO EM LOTE
[OK] Funcoes de execucao em lote carregadas:
  - get_datasets_for_batch(chunk_size_name, batch_name)
  - run_cdcms_batch(chunk_size_name, batch_name)
  - run_cdcms_all_batches(chunk_size_name)


In [ ]:
# =============================================================================
# CELULA 7.5: Teste Rapido (3 datasets)
# =============================================================================
# Executar em alguns datasets para validar que tudo funciona

print("="*70)
print("CELULA 7.5: TESTE RAPIDO")
print("="*70)

# Datasets de teste (representativos)
TEST_DATASETS = [
    ('SEA_Abrupt_Simple', 'chunk_500', 'batch_1'),
    ('HYPERPLANE_Abrupt_Simple', 'chunk_500', 'batch_1'),
    ('AGRAWAL_Abrupt_Simple_Mild', 'chunk_500', 'batch_1'),
]

print(f"Executando teste em {len(TEST_DATASETS)} datasets...")
print()

test_results = []

for dataset, chunk_size, batch in TEST_DATASETS:
    result = run_cdcms_on_dataset(
        dataset,
        chunk_size,
        batch,
        save_results=True,  # Salvar para validacao
        verbose=True
    )

    if result:
        test_results.append({
            'dataset': dataset,
            'prequential_gmean': result['avg_prequential_gmean'],
            'holdout_gmean': result['avg_holdout_gmean'],
            'time': result['execution_time_seconds']
        })

if test_results:
    print("\n" + "="*70)
    print("RESULTADOS DO TESTE")
    print("="*70)
    test_df = pd.DataFrame(test_results)
    print(test_df.to_string(index=False))
    print("\n[OK] Teste concluido com sucesso!")
else:
    print("\n[ERRO] Nenhum resultado obtido. Verifique:")
    print("  1. JARs do CDCMS (celulas 3 e 4)")
    print("  2. CDCMSEvaluator compilado (celula 5)")
    print("  3. Paths dos dados")

CELULA 7.5: TESTE RAPIDO
Executando teste em 3 datasets...


--- SEA_Abrupt_Simple (chunk_500/batch_1) ---
  [CONVERTIDO] Classe: bool -> int
  Instancias: 12000 | Chunks: 24 | Features: 3
  Executando CDCMS_CIL_GMean...
  [OK] Tempo: 60.3s
       Prequential G-Mean: 0.9410
       Holdout G-Mean:     0.9412
  [SALVO] /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/experiments_unified/chunk_500/batch_1/SEA_Abrupt_Simple/cdcms_results

--- HYPERPLANE_Abrupt_Simple (chunk_500/batch_1) ---
  Instancias: 12000 | Chunks: 24 | Features: 10
  Executando CDCMS_CIL_GMean...
  [OK] Tempo: 33.8s
       Prequential G-Mean: 0.9017
       Holdout G-Mean:     0.9039
  [SALVO] /content/drive/Othercomputers/Laptop-CIn/Downloads/DSL-AG-hybrid/experiments_unified/chunk_500/batch_1/HYPERPLANE_Abrupt_Simple/cdcms_results

--- AGRAWAL_Abrupt_Simple_Mild (chunk_500/batch_1) ---
  Instancias: 12000 | Chunks: 24 | Features: 9
  Executando CDCMS_CIL_GMean...
  [OK] Tempo: 65.1s
       Prequentia

In [ ]:
# =============================================================================
# CELULA 7.6: Executar em chunk_500 (Todos os Batches)
# =============================================================================
# AVISO: Isso pode demorar bastante (~85 datasets)

print("="*70)
print("CELULA 7.6: EXECUTAR CDCMS EM chunk_500")
print("="*70)

# Descomente para executar:
results_chunk500 = run_cdcms_all_batches('chunk_500_penalty', skip_existing=True)

# Ou execute batch por batch:
# results_b1 = run_cdcms_batch('chunk_500', 'batch_1')
# results_b2 = run_cdcms_batch('chunk_500', 'batch_2')
# results_b3 = run_cdcms_batch('chunk_500', 'batch_3')

print("Para executar, descomente uma das opcoes acima.")
print()
print("Opcao 1 - Todos os batches:")
print("  results_chunk500 = run_cdcms_all_batches('chunk_500', skip_existing=True)")
print()
print("Opcao 2 - Batch por batch:")
print("  results_b1 = run_cdcms_batch('chunk_500', 'batch_1')")


CELULA 7.6: EXECUTAR CDCMS EM chunk_500
Para executar, descomente uma das opcoes acima.

Opcao 1 - Todos os batches:
  results_chunk500 = run_cdcms_all_batches('chunk_500', skip_existing=True)

Opcao 2 - Batch por batch:
  results_b1 = run_cdcms_batch('chunk_500', 'batch_1')


In [ ]:
# =============================================================================
# CELULA 7.7: Executar em chunk_1000 (Todos os Batches)
# =============================================================================
# AVISO: Isso pode demorar bastante (~93 datasets)

print("="*70)
print("CELULA 7.7: EXECUTAR CDCMS EM chunk_1000")
print("="*70)

# Descomente para executar:
results_chunk1000 = run_cdcms_all_batches('chunk_1000_penalty', skip_existing=True)

# Ou execute batch por batch:
# results_b1 = run_cdcms_batch('chunk_1000', 'batch_1')
# results_b2 = run_cdcms_batch('chunk_1000', 'batch_2')
# results_b3 = run_cdcms_batch('chunk_1000', 'batch_3')
# results_b4 = run_cdcms_batch('chunk_1000', 'batch_4')

print("Para executar, descomente uma das opcoes acima.")
print()
print("Opcao 1 - Todos os batches:")
print("  results_chunk1000 = run_cdcms_all_batches('chunk_1000', skip_existing=True)")
print()
print("Opcao 2 - Batch por batch:")
print("  results_b1 = run_cdcms_batch('chunk_1000', 'batch_1')")



CELULA 7.7: EXECUTAR CDCMS EM chunk_1000
Para executar, descomente uma das opcoes acima.

Opcao 1 - Todos os batches:
  results_chunk1000 = run_cdcms_all_batches('chunk_1000', skip_existing=True)

Opcao 2 - Batch por batch:
  results_b1 = run_cdcms_batch('chunk_1000', 'batch_1')


In [ ]:
# =============================================================================
# CELULA 7.8: Consolidar e Comparar com EGIS
# =============================================================================
# Funcoes para consolidar resultados e comparar com EGIS

print("="*70)
print("CELULA 7.8: CONSOLIDAR E COMPARAR COM EGIS")
print("="*70)

def load_egis_results(chunk_size_name: str, batch_name: str, dataset_name: str) -> Optional[Dict]:
    """
    Carrega resultados do EGIS para um dataset.
    """
    metrics_file = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset_name / 'run_1' / 'chunk_metrics.json'

    if not metrics_file.exists():
        return None

    with open(metrics_file) as f:
        chunk_metrics = json.load(f)

    # Calcular media do test_gmean (chunks >= 1)
    test_gmeans = [m['test_gmean'] for m in chunk_metrics if m.get('test_gmean') is not None]

    if not test_gmeans:
        return None

    return {
        'avg_test_gmean': np.mean(test_gmeans),
        'std_test_gmean': np.std(test_gmeans),
        'num_chunks': len(chunk_metrics)
    }


def load_cdcms_results(chunk_size_name: str, batch_name: str, dataset_name: str) -> Optional[Dict]:
    """
    Carrega resultados do CDCMS para um dataset.
    """
    metrics_file = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset_name / 'cdcms_results' / 'chunk_metrics.json'

    if not metrics_file.exists():
        return None

    with open(metrics_file) as f:
        chunk_metrics = json.load(f)

    # Metricas prequential
    prequential_gmeans = [m['prequential_gmean'] for m in chunk_metrics]

    # Metricas holdout (chunks >= 1)
    holdout_gmeans = [m['holdout_gmean'] for m in chunk_metrics if m.get('holdout_gmean') is not None]

    return {
        'avg_prequential_gmean': np.mean(prequential_gmeans),
        'avg_holdout_gmean': np.mean(holdout_gmeans) if holdout_gmeans else None,
        'num_chunks': len(chunk_metrics)
    }


def compare_egis_cdcms(chunk_size_name: str) -> pd.DataFrame:
    """
    Compara resultados EGIS vs CDCMS para um chunk_size.
    """
    print(f"\nComparando EGIS vs CDCMS para {chunk_size_name}...")

    results = []
    batches = BATCHES.get(chunk_size_name, [])

    for batch in batches:
        datasets = get_datasets_for_batch(chunk_size_name, batch)

        for dataset in datasets:
            egis = load_egis_results(chunk_size_name, batch, dataset)
            cdcms = load_cdcms_results(chunk_size_name, batch, dataset)

            if egis and cdcms:
                results.append({
                    'dataset': dataset,
                    'batch': batch,
                    'egis_gmean': egis['avg_test_gmean'],
                    'cdcms_prequential': cdcms['avg_prequential_gmean'],
                    'cdcms_holdout': cdcms['avg_holdout_gmean'],
                    'diff_prequential': cdcms['avg_prequential_gmean'] - egis['avg_test_gmean'],
                    'diff_holdout': (cdcms['avg_holdout_gmean'] - egis['avg_test_gmean']) if cdcms['avg_holdout_gmean'] else None
                })

    if results:
        df = pd.DataFrame(results)

        print(f"\nDatasets comparados: {len(df)}")
        print(f"\nMedias:")
        print(f"  EGIS G-Mean:           {df['egis_gmean'].mean():.4f}")
        print(f"  CDCMS Prequential:     {df['cdcms_prequential'].mean():.4f}")
        if df['cdcms_holdout'].notna().any():
            print(f"  CDCMS Holdout:         {df['cdcms_holdout'].mean():.4f}")

        print(f"\nDiferencas (CDCMS - EGIS):")
        print(f"  Prequential: {df['diff_prequential'].mean():+.4f}")
        if df['diff_holdout'].notna().any():
            print(f"  Holdout:     {df['diff_holdout'].mean():+.4f}")

        return df

    return pd.DataFrame()


def generate_comparison_table(chunk_size_name: str) -> None:
    """
    Gera tabela de comparacao formatada.
    """
    df = compare_egis_cdcms(chunk_size_name)

    if df.empty:
        print("Sem dados para comparacao.")
        return

    # Salvar CSV
    output_file = EXPERIMENTS_DIR / chunk_size_name / f'comparison_egis_cdcms_{chunk_size_name}.csv'
    df.to_csv(output_file, index=False)
    print(f"\n[SALVO] {output_file}")

    # Top 5 onde CDCMS e melhor
    print("\nTop 5 - CDCMS melhor que EGIS (holdout):")
    if df['diff_holdout'].notna().any():
        top5_cdcms = df.nlargest(5, 'diff_holdout')[['dataset', 'egis_gmean', 'cdcms_holdout', 'diff_holdout']]
        print(top5_cdcms.to_string(index=False))

    # Top 5 onde EGIS e melhor
    print("\nTop 5 - EGIS melhor que CDCMS (holdout):")
    if df['diff_holdout'].notna().any():
        top5_egis = df.nsmallest(5, 'diff_holdout')[['dataset', 'egis_gmean', 'cdcms_holdout', 'diff_holdout']]
        print(top5_egis.to_string(index=False))


print("[OK] Funcoes de comparacao carregadas:")
print("  - load_egis_results(chunk_size_name, batch_name, dataset_name)")
print("  - load_cdcms_results(chunk_size_name, batch_name, dataset_name)")
print("  - compare_egis_cdcms(chunk_size_name)")
print("  - generate_comparison_table(chunk_size_name)")
print()
print("Para gerar comparacao:")
print("  generate_comparison_table('chunk_500')")
print("  generate_comparison_table('chunk_1000')")


# =============================================================================
# FIM DO ARQUIVO
# =============================================================================
print("\n" + "="*70)
print("CDCMS PARTE 7 COMPLETA - CARREGADA COM SUCESSO!")
print("="*70)
print()
print("Proximos passos:")
print("  1. Execute CELULA 7.5 para teste rapido")
print("  2. Execute CELULA 7.6 para chunk_500")
print("  3. Execute CELULA 7.7 para chunk_1000")
print("  4. Execute CELULA 7.8 para comparar com EGIS")


CELULA 7.8: CONSOLIDAR E COMPARAR COM EGIS
[OK] Funcoes de comparacao carregadas:
  - load_egis_results(chunk_size_name, batch_name, dataset_name)
  - load_cdcms_results(chunk_size_name, batch_name, dataset_name)
  - compare_egis_cdcms(chunk_size_name)
  - generate_comparison_table(chunk_size_name)

Para gerar comparacao:
  generate_comparison_table('chunk_500')
  generate_comparison_table('chunk_1000')

CDCMS PARTE 7 COMPLETA - CARREGADA COM SUCESSO!

Proximos passos:
  1. Execute CELULA 7.5 para teste rapido
  2. Execute CELULA 7.6 para chunk_500
  3. Execute CELULA 7.7 para chunk_1000
  4. Execute CELULA 7.8 para comparar com EGIS


## PARTE 8: Resumo Final

In [ ]:
# =============================================================================
# CELULA 7.9: Gerar Relatorio Consolidado EGIS vs CDCMS
# =============================================================================
# Gera tabela comparativa com tratamento adequado para datasets multiclasse
# =============================================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime

print("="*70)
print("CELULA 7.9: GERAR RELATORIO CONSOLIDADO EGIS vs CDCMS")
print("="*70)

# =============================================================================
# 1. Funcao para carregar resultados EGIS
# =============================================================================

def load_egis_results_detailed(chunk_size_name: str, batch_name: str, dataset_name: str) -> dict:
    """Carrega resultados detalhados do EGIS."""
    metrics_file = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset_name / 'run_1' / 'chunk_metrics.json'

    if not metrics_file.exists():
        return None

    with open(metrics_file) as f:
        chunk_metrics = json.load(f)

    # Calcular medias
    test_gmeans = [m.get('test_gmean') for m in chunk_metrics if m.get('test_gmean') is not None]
    test_f1s = [m.get('test_f1') for m in chunk_metrics if m.get('test_f1') is not None]

    if not test_gmeans:
        return None

    return {
        'avg_test_gmean': sum(test_gmeans) / len(test_gmeans),
        'avg_test_f1': sum(test_f1s) / len(test_f1s) if test_f1s else None,
        'num_chunks': len(chunk_metrics)
    }


# =============================================================================
# 2. Funcao para carregar resultados CDCMS
# =============================================================================

def load_cdcms_results_detailed(chunk_size_name: str, batch_name: str, dataset_name: str) -> dict:
    """Carrega resultados detalhados do CDCMS."""

    # Verificar se e multiclasse primeiro
    if dataset_name in MULTICLASS_DATASETS:
        return {
            'status': 'N/A',
            'reason': f'Multiclass ({MULTICLASS_DATASETS[dataset_name]} classes)',
            'avg_prequential_gmean': None,
            'avg_holdout_gmean': None,
            'avg_prequential_f1': None,
            'avg_holdout_f1': None
        }

    metrics_file = EXPERIMENTS_DIR / chunk_size_name / batch_name / dataset_name / 'cdcms_results' / 'chunk_metrics.json'

    if not metrics_file.exists():
        return {
            'status': 'NOT_RUN',
            'reason': 'Results not found',
            'avg_prequential_gmean': None,
            'avg_holdout_gmean': None
        }

    with open(metrics_file) as f:
        chunk_metrics = json.load(f)

    # Calcular medias
    prequential_gmeans = [m['prequential_gmean'] for m in chunk_metrics]
    holdout_gmeans = [m['holdout_gmean'] for m in chunk_metrics if m.get('holdout_gmean') is not None]

    return {
        'status': 'OK',
        'avg_prequential_gmean': sum(prequential_gmeans) / len(prequential_gmeans),
        'avg_holdout_gmean': sum(holdout_gmeans) / len(holdout_gmeans) if holdout_gmeans else None
    }


# =============================================================================
# 3. Gerar tabela comparativa completa
# =============================================================================

def generate_comparison_table(chunk_size_name: str) -> pd.DataFrame:
    """
    Gera tabela comparativa EGIS vs CDCMS para um chunk_size.
    Inclui marcacao N/A para datasets multiclasse.
    """
    print(f"\nGerando tabela comparativa para {chunk_size_name}...")

    results = []
    batches = BATCHES.get(chunk_size_name, [])

    for batch in batches:
        datasets = get_datasets_for_batch(chunk_size_name, batch)

        for dataset in datasets:
            egis = load_egis_results_detailed(chunk_size_name, batch, dataset)
            cdcms = load_cdcms_results_detailed(chunk_size_name, batch, dataset)

            row = {
                'dataset': dataset,
                'batch': batch,
                'chunk_size': chunk_size_name,
            }

            # EGIS results
            if egis:
                row['egis_gmean'] = egis['avg_test_gmean']
                row['egis_f1'] = egis.get('avg_test_f1')
            else:
                row['egis_gmean'] = None
                row['egis_f1'] = None

            # CDCMS results
            if cdcms:
                row['cdcms_status'] = cdcms.get('status', 'OK')
                row['cdcms_prequential_gmean'] = cdcms.get('avg_prequential_gmean')
                row['cdcms_holdout_gmean'] = cdcms.get('avg_holdout_gmean')
                row['cdcms_note'] = cdcms.get('reason', '')
            else:
                row['cdcms_status'] = 'NOT_RUN'
                row['cdcms_prequential_gmean'] = None
                row['cdcms_holdout_gmean'] = None
                row['cdcms_note'] = ''

            # Calcular diferenca (apenas para datasets binarios com resultados)
            if (row['cdcms_status'] == 'OK' and
                row['egis_gmean'] is not None and
                row['cdcms_holdout_gmean'] is not None):
                row['diff_holdout'] = row['cdcms_holdout_gmean'] - row['egis_gmean']
            else:
                row['diff_holdout'] = None

            results.append(row)

    df = pd.DataFrame(results)
    return df


# =============================================================================
# 4. Gerar relatorio formatado
# =============================================================================

def generate_full_report(chunk_size_name: str):
    """Gera relatorio completo com estatisticas."""

    df = generate_comparison_table(chunk_size_name)

    if df.empty:
        print("Sem dados para gerar relatorio.")
        return

    # Separar por status
    binary_ok = df[df['cdcms_status'] == 'OK']
    multiclass = df[df['cdcms_status'] == 'N/A']
    not_run = df[df['cdcms_status'] == 'NOT_RUN']

    print("\n" + "="*70)
    print(f"RELATORIO CONSOLIDADO: {chunk_size_name}")
    print("="*70)

    print(f"\nTotal de datasets: {len(df)}")
    print(f"  - Binarios (comparaveis): {len(binary_ok)}")
    print(f"  - Multiclasse (N/A):      {len(multiclass)}")
    print(f"  - Nao executados:         {len(not_run)}")

    # Estatisticas apenas para datasets binarios
    if not binary_ok.empty:
        print("\n" + "-"*50)
        print("ESTATISTICAS (apenas datasets BINARIOS)")
        print("-"*50)

        print(f"\nEGIS:")
        print(f"  Media G-Mean: {binary_ok['egis_gmean'].mean():.4f}")

        print(f"\nCDCMS:")
        print(f"  Media Prequential G-Mean: {binary_ok['cdcms_prequential_gmean'].mean():.4f}")
        if binary_ok['cdcms_holdout_gmean'].notna().any():
            print(f"  Media Holdout G-Mean:     {binary_ok['cdcms_holdout_gmean'].mean():.4f}")

        # Comparacao
        valid_diff = binary_ok['diff_holdout'].dropna()
        if not valid_diff.empty:
            print(f"\nComparacao (CDCMS Holdout - EGIS):")
            print(f"  Media:  {valid_diff.mean():+.4f}")
            print(f"  Mediana: {valid_diff.median():+.4f}")

            # Contagem de vitorias
            cdcms_wins = (valid_diff > 0).sum()
            egis_wins = (valid_diff < 0).sum()
            ties = (valid_diff == 0).sum()
            print(f"\n  CDCMS melhor: {cdcms_wins}")
            print(f"  EGIS melhor:  {egis_wins}")
            print(f"  Empates:      {ties}")

    # Listar datasets multiclasse
    if not multiclass.empty:
        print("\n" + "-"*50)
        print("DATASETS MULTICLASSE (excluidos da comparacao)")
        print("-"*50)
        for _, row in multiclass.iterrows():
            print(f"  - {row['dataset']}: {row['cdcms_note']}")

    # Salvar CSV
    output_dir = EXPERIMENTS_DIR / chunk_size_name
    output_dir.mkdir(exist_ok=True)

    # CSV completo
    csv_file = output_dir / f'comparison_egis_cdcms_{chunk_size_name}_full.csv'
    df.to_csv(csv_file, index=False)
    print(f"\n[SALVO] {csv_file}")

    # CSV apenas binarios
    if not binary_ok.empty:
        csv_binary = output_dir / f'comparison_egis_cdcms_{chunk_size_name}_binary_only.csv'
        binary_ok.to_csv(csv_binary, index=False)
        print(f"[SALVO] {csv_binary}")

    return df


# =============================================================================
# 5. Executar
# =============================================================================

print("\nFuncoes disponiveis:")
print("  - generate_comparison_table(chunk_size_name)")
print("  - generate_full_report(chunk_size_name)")
print()
print("Para gerar relatorio:")
print("  df_500 = generate_full_report('chunk_500')")
print("  df_1000 = generate_full_report('chunk_1000')")


CELULA 7.9: GERAR RELATORIO CONSOLIDADO EGIS vs CDCMS

Funcoes disponiveis:
  - generate_comparison_table(chunk_size_name)
  - generate_full_report(chunk_size_name)

Para gerar relatorio:
  df_500 = generate_full_report('chunk_500')
  df_1000 = generate_full_report('chunk_1000')
